# Create a hold-out set for final testing

Author: Pete King

This notebook creates a separate testing dataset to hold in reserve for a final evaluation of our selected model's ability to generalize to unseen data.

In [1]:
#123456789012345678901234567890123456789012345678901234567890123456789012345678
import re

import pandas as pd

import data_prep as dp

In [2]:
# Select the data file and start date for the testing set
# ----------------------------------------------------------------------------
# We want to rigorously test our final model against a period of dynamic
# real-world volatility.  For example, beginning in October 2024 would capture
# an initial period of relatively low volatility, followed by the highly 
# volatility period after Trump announced reciprocal tarrifs in April 2025.
# ----------------------------------------------------------------------------
DATA_FILENAME = 'raw_data_prediction_dataset.csv'
TESTING_START_DATE = '2024-10-01'
# Select buffer size between validation/test and training sets
# We calculate target variables using 63 days of "future" historical data
# A buffer of one quarter (63 business days) guards against data leakage
BUFFER_SIZE = 63

In [3]:
# Import feature dataset with labels
df = pd.read_csv(DATA_FILENAME)
df

,date,nominal_GDP,real_GDP,debt_to_GDP,debt_interest,consumer_price_index,core_PCI,personal_consumption_expenditure,core_PCE,producer_price_index,...,XLP_volatility_target,XLY_volatility_target,XLRE_volatility_target,BIL_volatility_target,IEF_volatility_target,TLT_volatility_target,LQD_volatility_target,HYG_volatility_target,TIP_volatility_target,GLD_volatility_target
0,1927-12-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1928-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1928-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1928-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1928-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24950,2026-03-12,31442.483,24065.956,122.49035,1227.495,327.46,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24951,2026-03-13,31442.483,24065.956,122.49035,1227.495,327.46,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24952,2026-03-16,31442.483,24065.956,122.49035,1227.495,327.46,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24953,2026-03-17,31442.483,24065.956,122.49035,1227.495,327.46,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Let's look at the earliest starting dates for each of our column labels to see how far back our labels go for each ETF.

In [4]:
label_tag = '_volatility_target'
# Use a regular expression to parse the ticker symbol from the column name
p = re.compile(label_tag)
start_dates = {}
for column_name in df.columns:
    if label_tag in column_name:
        ticker = p.split(column_name)[0]
        start_dates[ticker] = (
            df[['date', column_name]].dropna().date.iloc[0]
        )
sorted([(k, v) for k, v in start_dates.items()], key=lambda pair: pair[1])

[('XLF', '1999-02-16'),
 ('XLK', '1999-02-16'),
 ('XLU', '1999-02-16'),
 ('XLV', '1999-02-16'),
 ('XLE', '1999-02-16'),
 ('XLI', '1999-02-16'),
 ('XLB', '1999-02-16'),
 ('XLP', '1999-02-16'),
 ('XLY', '1999-02-16'),
 ('IEF', '2003-02-18'),
 ('TLT', '2003-02-18'),
 ('LQD', '2003-02-18'),
 ('TIP', '2004-02-17'),
 ('GLD', '2005-02-22'),
 ('BIL', '2008-02-19'),
 ('HYG', '2008-02-19'),
 ('XLRE', '2016-02-16')]

In [5]:
print(f'The XLRE (Real Estate) series only goes back to {start_dates['XLRE']}')

The XLRE (Real Estate) series only goes back to 2016-02-16


For the Vector AutoRegressive Integrated Moving Average (VARIMA) and the multivariate Long Short-Term Memory (LSTM) models, we intend to use fluctuations in the price of all labeled ETF price time series as features.  However, we've observed that a few of these time series have significantly later start dates than the others.  For example, if we drop the 'XLRE' ETF (Real Estate) from the dataset, we'll free up 8 years worth of training data for the rest of the ETFs.

Going forward, we plan to test three variations of the data:

 - Including all raw data from roughly 2015 (start of XLRE data)
 - Dropping XLRE (Real Estate) to make data as early as 2007 available
 - Dropping HYG and BIL to make data as early as 2004 available

In [6]:
train_val_df, test_df = dp.time_series_split(
    df, TESTING_START_DATE, BUFFER_SIZE
)
train_val_df

,date,nominal_GDP,real_GDP,debt_to_GDP,debt_interest,consumer_price_index,core_PCI,personal_consumption_expenditure,core_PCE,producer_price_index,...,XLP_volatility_target,XLY_volatility_target,XLRE_volatility_target,BIL_volatility_target,IEF_volatility_target,TLT_volatility_target,LQD_volatility_target,HYG_volatility_target,TIP_volatility_target,GLD_volatility_target
0,1927-12-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1928-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1928-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1928-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1928-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24510,2024-06-27,29147.044,23286.508,119.50314,1103.565,313.044,318.39,123.539,122.677,144.869,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24511,2024-06-28,29147.044,23286.508,119.50314,1103.565,313.044,318.39,123.539,122.677,144.869,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24512,2024-07-01,29511.664,23478.570,120.17172,1146.182,313.569,318.94,123.736,122.911,144.922,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24513,2024-07-02,29511.664,23478.570,120.17172,1146.182,313.569,318.94,123.736,122.911,144.922,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
test_df

,date,nominal_GDP,real_GDP,debt_to_GDP,debt_interest,consumer_price_index,core_PCI,personal_consumption_expenditure,core_PCE,producer_price_index,...,XLP_volatility_target,XLY_volatility_target,XLRE_volatility_target,BIL_volatility_target,IEF_volatility_target,TLT_volatility_target,LQD_volatility_target,HYG_volatility_target,TIP_volatility_target,GLD_volatility_target
24578,2024-10-01,29825.182,23586.542,121.43633,1155.616,315.631,321.731,124.494,123.832,146.294,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24579,2024-10-02,29825.182,23586.542,121.43633,1155.616,315.631,321.731,124.494,123.832,146.294,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24580,2024-10-03,29825.182,23586.542,121.43633,1155.616,315.631,321.731,124.494,123.832,146.294,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24581,2024-10-04,29825.182,23586.542,121.43633,1155.616,315.631,321.731,124.494,123.832,146.294,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24582,2024-10-07,29825.182,23586.542,121.43633,1155.616,315.631,321.731,124.494,123.832,146.294,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24950,2026-03-12,31442.483,24065.956,122.49035,1227.495,327.460,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24951,2026-03-13,31442.483,24065.956,122.49035,1227.495,327.460,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24952,2026-03-16,31442.483,24065.956,122.49035,1227.495,327.460,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24953,2026-03-17,31442.483,24065.956,122.49035,1227.495,327.460,333.512,128.969,128.394,153.231,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The gap between the training and testing sets guards against data leakage.

In [8]:
# Save segmented datasets for further analysis; drop the index
train_val_df.to_csv('train_val_data.csv', index=False)
test_df.to_csv('test_data.csv', index=False)